# slopes calculation
- input
    - bootstrap samples with annual means from 70_xx
- slope within each bootstrap sample from linear regression to annual means (property against environmental temperature)
- mean across samples
- relative slope by dividing through multiannual mean
- CI based on bootstrapped percentiles
- output:
    - one csv for each of the 4 dataset with relative slopes and their CI 

In [1]:
import os 
import pickle
import numpy as np
import xarray as xr

from pathlib import Path
import sys

sys.path.append("../tools")


from cell_props import calc_slope_perT_fromerr # function for slopes

In [2]:
input_dir = Path("../output/annual/data")
output_directory = Path("../output/annual/data_slopes")
output_directory.mkdir(parents=True, exist_ok=True)


years = [2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032,
       2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
       2045, 2046, 2047, 2048, 2049]

subsets=  ["cell_tracks","cell_features","anvil_features", "domain"]


In [3]:
for name_subset in subsets:
    ######################################################
    ## open data
    # load variable data and select year
    input_path = os.path.join(input_dir, f"yearly_stats_abso_boot_{name_subset}.pkl")
    with open(input_path, "rb") as f:
        yearly_stats = pickle.load(f)

    # select all non-temperature variables
    non_tas_vars = [v for v in yearly_stats["mean"].data_vars if "tas" not in v]
    
    var_means = yearly_stats["mean"][non_tas_vars].sel(year=years)
    var_boot = yearly_stats["boot"][non_tas_vars].sel(year=years)
    
    # select the (single) temperature variable
    tas_var = next(v for v in yearly_stats["mean"].data_vars if "tas" in v)
    
    t_means = yearly_stats["mean"][tas_var].sel(year=years)
    t_boot = yearly_stats["boot"][tas_var].sel(year=years)

    ######################################################
    # slope calculation
    
    df_slopes, df_all_slopes, df_all_intercepts = calc_slope_perT_fromerr(
        var_means,
        t_means,
        var_boot,
        t_boot,
        alpha=0.1)

    # Add years that are included in slpoe calc
    df_slopes["year_min"] = years[0]
    df_slopes["year_max"] = years[-1]

    ######################################################
    ## save absolute slopes 

    # save all raw bootstrap slopes ABSOLUTE
    df_all_slopes.to_csv(
        os.path.join(output_directory,f"df_lm_slopes_all_boot_{name_subset}_rel_T_CI_space.csv"),index=True)
    
    # save df all interceps ABOSLUTE
    df_all_intercepts.to_csv(
        os.path.join(output_directory, f"df_lm_intercepts_all_boot_{name_subset}_rel_T_CI_space.csv"), index=True)

    # save summary df slopes ABOSLUTE
    df_slopes.to_csv(
        os.path.join(output_directory, f"df_lm_coefficients_boot_{name_subset}_rel_T_CIspace.csv"),index=False)

    ######################################################
    # get relative slopes of df slopes 
    ## get baseline
    base = {}
    for var in var_means.data_vars:
        base[var] = float(var_means[var].mean())
        
    ## select columns that should be converted
    slope_vars = [col for col in df_slopes.columns if "slope" in col or "intercept" in col]
    
    ## create new df
    df_slopes_rel = df_slopes.copy()

    ## devide selected columns by baseline
    for var in var_means.data_vars:
        mask = df_slopes_rel["variable"] == var
        df_slopes_rel.loc[mask, slope_vars] = df_slopes.loc[mask, slope_vars] / base[var]

    #ä save  summary to csv
    df_slopes_rel.to_csv(
        os.path.join(output_directory,
                     f"df_lm_coefficients_boot_{name_subset}_rel_T_relative_CIspace.csv"),
        index=False
    )

    # convert all slopes =>  all raw bootstrap slopes relative to each variable baseline
    ## new df
    df_all_slopes_rel = df_all_slopes
    df_all_intercepts_rel = df_all_intercepts

    ## devide slopesby baseline
    for var in df_all_slopes_rel.columns:
        if var in base:  # only normalize the variables that have a baseline
            df_all_slopes_rel[var] = df_all_slopes_rel[var] / base[var]
            df_all_intercepts_rel[var] = df_all_intercepts_rel[var] / base[var]
            
    ## save all slopes
    df_all_slopes_rel.to_csv(
        os.path.join(
            output_directory,
            f"df_lm_slopes_all_boot_{name_subset}_rel_T_relative_CIspace.csv"
        ),
        index=True
    )

    ## save all intercepts RELATIVE
    df_all_intercepts_rel.to_csv(
        os.path.join(
            output_directory,
            f"df_lm_intercepts_all_boot_{name_subset}_rel_T_relative_CIspace.csv"
        ),
        index=True
    )



In [7]:
tmp = [col for col in df_slopes.columns if "slope" in col or "intercept" in col]
tmp

['slope',
 'intercept',
 'slope_ci_lower',
 'slope_ci_upper',
 'intercept_ci_lower',
 'intercept_ci_upper']

In [8]:
df_slopes

,variable,slope,intercept,slope_ci_lower,slope_ci_upper,intercept_ci_lower,intercept_ci_upper,year_min,year_max
0,n_cells,7.956715e-02,9.534568e+01,-4.188506e+00,4.099189e+00,1.369606e+03,-1.104926e+03,2021,2049
1,n_anvils,-2.868938e-01,1.577601e+02,-2.629653e+00,2.166388e+00,8.576653e+02,-5.746502e+02,2021,2049
2,cell_area_sum,7.919747e+09,-2.218710e+12,-1.735908e+09,1.911623e+10,6.665479e+11,-5.561785e+12,2021,2049
3,anvil_area_sum,-6.591418e+09,6.096344e+12,-1.812179e+11,1.797075e+11,5.821115e+13,-4.954430e+13,2021,2049
4,rain_volume_cell_sum,2.371216e+11,-6.801922e+13,3.385897e+10,4.522218e+11,-7.325731e+12,-1.322410e+14,2021,2049
5,rain_volume_anvil_sum,3.276280e+11,-8.956096e+13,-1.009399e+11,7.975167e+11,3.841418e+13,-2.298512e+14,2021,2049
